In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
import linear # type: ignore
import binary_cross_entropy # type: ignore
from common import assert_eq, average_run, test_perceptron_boolean # type: ignore


class Per_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, y):
        z = linear.linear(x, W, b)
        p = sigmoid.sigmoid(z)
        L = binary_cross_entropy.binary_cross_entropy(p, y)
        F = L

        ctx.save_for_backward(x, W, b, y, p)

        return F

    @staticmethod
    def backward(ctx, dF_dL):
        (x, W, b, y, p) = ctx.saved_tensors

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out / Neurons

        assert_eq(x.shape, (S, FI))
        assert_eq(W.shape, (FI, FO))
        assert_eq(b.shape, (FO,))
        assert_eq(y.shape, (S, FO))
        assert_eq(p.shape, (S, FO))

        # z = xW + b
        # p = e^z / (e^z + 1)
        # L = -y log(p) - (1-y) log(1-p)

        dz_dx = W
        assert_eq(dz_dx.shape, (FI, FO))

        dz_dW = x
        assert_eq(dz_dW.shape, (S, FI))

        dz_db = 1
        assert_eq(dz_db, 1)

        dp_dz = p * (1 - p)
        assert_eq(dp_dz.shape, (S, FO))

        dp_dx = dp_dz * dz_dx.t()
        assert_eq(dp_dx.shape, (S, FI))

        dp_dW = dp_dz * dz_dW
        assert_eq(dp_dW.shape, (S, FI))

        dp_db = dp_dz * dz_db
        assert_eq(dp_db.shape, (S, FO))

        # average over samples
        dL_dp = (p - y) / (p * (1 - p)) / S
        assert_eq(dL_dp.shape, (S, FO))

        dL_dz = dL_dp * dp_dz
        assert_eq(dL_dz.shape, (S, FO))

        # (samples, features_in) = (samples, features_out) @ (features_out, features_in)
        dL_dx = dL_dz @ dz_dx.t()
        assert_eq(dL_dx.shape, (S, FI))
        
        # (features_in, features_out) = (features_in, samples) @ (samples, features_out)
        dL_dW = dz_dW.t() @ dL_dz
        assert_eq(dL_dW.shape, (FI, FO))

        dL_db = dL_dz.sum(dim=0)
        assert_eq(dL_db.shape, (FO,))

        dL_dy = None
        assert_eq(dL_dy is None, True)

        dF_dx = dF_dL * dL_dx
        assert_eq(dF_dx.shape, (S, FI))

        dF_dW = dF_dL * dL_dW
        assert_eq(dF_dW.shape, (FI, FO))

        dF_db = dF_dL * dL_db
        assert_eq(dF_db.shape, (FO,))

        dF_dy = dL_dy
        assert_eq(dF_dy, None)

        return (dF_dx, dF_dW, dF_db, dF_dy)
    

class Per_Lin_Sig_BCE_Gradient(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def forward(self, x, y):
        return Per_Lin_Sig_BCE_Gradient_Function.apply(x, self.weight.T, self.bias, y)
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def learn(self, x, y):
        return self(x, y)

    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def predict(self, x):
        with tr.no_grad():
            return sigmoid.SigmoidFunction.apply(linear.LinearFunction.apply(x, self.weight.T, self.bias)) >= 0.5


def test_Per_Lin_Sig_BCE_Gradient(bool_fn, epochs, lr=0.1):
    return test_perceptron_boolean(Per_Lin_Sig_BCE_Gradient, bool_fn, epochs=epochs, lr=lr)


if __name__ == "__main__":
    logical_and = tr.logical_and
    print(f" AND, {500} epochs: {average_run(100, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_and, epochs=500)):.2f}")

    logical_or = tr.logical_or
    print(f"  OR, {500} epochs: {average_run(100, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_or, epochs=500)):.2f}")

    logical_nand = lambda x1, x2: tr.logical_not(tr.logical_and(x1, x2))
    print(f"NAND, {500} epochs: {average_run(100, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_nand, epochs=500)):.2f}")

    logical_xor = lambda x1, x2: tr.logical_xor(x1, x2)
    print(f" XOR, {500} epochs: {average_run(100, lambda: test_Per_Lin_Sig_BCE_Gradient(logical_xor, epochs=500)):.2f}")


NameError: name 'tr' is not defined